In [ ]:
# For reducing the model's size to simplify building it. Optional, but make sure you know all potential conditions you'll be testing.
import cobra,json,os,sys,re,pandas as pd, io, requests,numpy as np
from bs4 import BeautifulSoup
from collections import OrderedDict
from copy import deepcopy

# keep going up until the "pycore" folder is found
wd = os.getcwd()
os.chdir('../../../../')
base_path = os.getcwd()
sys.path.append(base_path)
from pycore import *

os.chdir(wd)

from scipy.io import loadmat

# read MatLab file; extract res.fluxes
data = loadmat('res.mat')
fluxes = data['res']['fluxes'][0,0]
equations = data['model']['RxnFormula'][0,0]
rxn_ids = data['model']['rid'][0,0]
rxn_props = data['model']['rxnprop'][0,0]

# remove rxns where 'subsystem' under 'rxn_props' is 'dilution'
subsystems = rxn_props['subsystem']
valid_indices = [i for i in range(len(subsystems)) if subsystems[i] != 'dilution']
rxn_ids = [rxn_ids[i] for i in valid_indices]
equations = [equations[i] for i in valid_indices]

# combine rxn_ids and equations into a dictionary
eq_dict = {}
for i in valid_indices:
	rxn_id = rxn_ids[i][0][0]
	equation = equations[i][0][0]
	eq_dict[rxn_id] = equation

# make dataframe with columns 'id', 'type', 'val', 'vLB', 'vUB'
df = pd.DataFrame(fluxes[0], columns=['id', 'type', 'val', 'vLB', 'vUB'])
# convert all entries to their underlying values
df = df.applymap(lambda x: x.flatten()[0] if isinstance(x, np.ndarray) and x.size > 0 else x)
# reduce 'id' column lists to their 1st entries
df['id'] = df['id'].apply(lambda x: x[0] if isinstance(x, (list, np.ndarray)) and len(x) > 0 else x)
# save to TSV
df.to_csv('fluxes_raw.tsv', sep='\t', index=False)

# for each row, split the .f or .r at the end of the id, save to 'rxn_id', and save the suffix to 'suffix'
df['rxn_id'] = df['id'].apply(lambda x: re.sub(r'\.(f|r)$', '', x) if isinstance(x, str) else x)
# remove rows where 'rxn_id' is not in eq_dict (e.g., to remove dilution reactions)
df = df[df['rxn_id'].apply(lambda x: x in eq_dict)]
# add equations to dataframe in 'reaction' column; keep ID before the .f or .r suffix
df['reaction'] = df['rxn_id'].apply(lambda x: eq_dict[x] if x in eq_dict else None)
df['dir'] = df['id'].apply(lambda x: re.search(r'\.(f|r)$', x).group(1) if isinstance(x, str) and re.search(r'\.(f|r)$', x) else None)
gross_fluxes = pd.DataFrame(columns=df.columns.tolist())
net_fluxes = pd.DataFrame(columns=['rxn_id', 'reaction', 'val', 'vLB', 'vUB'])
# for each unique rxn_id, find entry with type 'net'. If there's no entry with type 'exc', then infer the direction from the sign of 'val' and the 'dir' column (change direction if val < 0)
for rxn_id in df['rxn_id'].unique():
	sub_df = df[df['rxn_id'] == rxn_id]
	# determine net flux from 'val' of entries with type 'net'
	net_rows = sub_df[sub_df['type'] == 'net'][['val', 'dir']]
	# calculate net flux as sum of 'val' entries with type 'net', inverting sign for 'r' directions
	net_flux = sum(net_rows.apply(lambda row: row['val'] if row['dir'] == 'f' else -row['val'], axis=1))
	# get vUB and vLB from the first net row in sub_df
	fwd_UB = sub_df[sub_df['type'] == 'net'].iloc[0]['vUB']
	rev_UB = -sub_df[sub_df['type'] == 'net'].iloc[0]['vLB']
	# if there's an entry with type 'exc', use that
	exc_rows = sub_df[sub_df['type'] == 'exc']
	if len(exc_rows) == 0:
		# assume fwd flux is net flux if net flux >= 0, else rev flux is net flux; the other direction is 0
		if net_flux >= 0:
			fwd_flux, rev_flux = net_flux, 0
		else:
			fwd_flux, rev_flux = 0, -net_flux
	else:
		# if the dir of the exc row is 'f', then fwd_flux is its val; if dir is 'r', then rev_flux is its val
		if len(exc_rows) == 1:
			if exc_rows.iloc[0]['dir'] == 'f':
				fwd_flux = exc_rows.iloc[0]['val']
				rev_flux = fwd_flux - net_flux
			else:
				rev_flux = exc_rows.iloc[0]['val']
				fwd_flux = rev_flux + net_flux
		else:
			raise ValueError(f"More than 1 'exc' row found for reaction {rxn_id}")
	# create new rows for fwd and rev fluxes
	fwd_row = {'id': f"{rxn_id}.f", 'type': 'exc', 'val': fwd_flux, 'vLB': 0, 'vUB': fwd_UB, 'rxn_id': rxn_id, 'dir': 'f', 'reaction': eq_dict[rxn_id] if rxn_id in eq_dict else None}
	rev_row = {'id': f"{rxn_id}.r", 'type': 'exc', 'val': rev_flux, 'vLB': 0, 'vUB': rev_UB, 'rxn_id': rxn_id, 'dir': 'r', 'reaction': eq_dict[rxn_id] if rxn_id in eq_dict else None}
	gross_fluxes = pd.concat([gross_fluxes, pd.DataFrame([fwd_row, rev_row])], ignore_index=True)
	# add net flux to net_fluxes dataframe
	net_fluxes = pd.concat([net_fluxes, pd.DataFrame([{'rxn_id': rxn_id, 'reaction': eq_dict[rxn_id] if rxn_id in eq_dict else None, 'val': net_flux, 'vLB': -rev_UB, 'vUB': fwd_UB}])], ignore_index=True)
# keep only columns 'rxn_id', 'dir', 'val', 'vLB', 'vUB'
gross_fluxes = gross_fluxes[['rxn_id', 'reaction', 'dir', 'val', 'vLB', 'vUB']]
# save to TSV
gross_fluxes.to_csv('fluxes_gross.tsv', sep='\t', index=False)
net_fluxes.to_csv('fluxes_net.tsv', sep='\t', index=False)

/var/folders/fx/q_x2rm651s91fj2m6_tb19pxvpxbmb/T/ipykernel_73613/1491914573.py:41: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.flatten()[0] if isinstance(x, np.ndarray) and x.size > 0 else x)
/var/folders/fx/q_x2rm651s91fj2m6_tb19pxvpxbmb/T/ipykernel_73613/1491914573.py:88: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  gross_fluxes = pd.concat([gross_fluxes, pd.DataFrame([fwd_row, rev_row])], ignore_index=True)
/var/folders/fx/q_x2rm651s91fj2m6_tb19pxvpxbmb/T/ipykernel_73613/1491914573.py:90: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determ